In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
WORK = '/kaggle/working'

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]

BASE = os.path.dirname(find('train.csv'))
test = pd.read_csv(f'{BASE}/test.csv')
folds = pd.read_csv(find('folds.csv'))
fg = pd.read_csv(find('folds_group.csv'))
folds = folds.merge(fg[['uid', 'g120', 'g250']], on='uid', how='left')
assert folds[['g120', 'g250']].isna().sum().sum() == 0
te_uids = ['test/' + f for f in test.filename]

TF = pd.read_csv(find('text_features.csv'), index_col='uid')
TF_tr, TF_te = TF.loc[folds.uid], TF.loc[te_uids]

def load_layer(emb_file, uid_file, layer):
    E = np.load(find(emb_file), mmap_mode='r')
    ids = pd.read_csv(find(uid_file)).uid.tolist()
    ix = {u: i for i, u in enumerate(ids)}
    A = np.asarray(E[:, layer, :], dtype=np.float32)
    return A[[ix[u] for u in folds.uid]], A[[ix[u] for u in te_uids]]

W_tr, W_te = load_layer('wavlm_large_mean.npy', 'wavlm_uids.csv', 21)
H_tr, H_te = load_layer('whisper_enc_mean.npy', 'whisper_enc_uids.csv', 30)

y = folds.label.values
w = folds.w_test.values
nz = (folds.is_zero == 0).values
grp = folds.dur_group.values

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def pear(a, b): return float(np.corrcoef(a, b)[0, 1])
def wrmse(a, b, ww): return float(np.sqrt(np.sum(ww * (a - b) ** 2) / np.sum(ww)))
def wpear(a, b, ww):
    ma, mb = np.average(a, weights=ww), np.average(b, weights=ww)
    return float(np.sum(ww * (a - ma) * (b - mb)) /
                 np.sqrt(np.sum(ww * (a - ma) ** 2) * np.sum(ww * (b - mb) ** 2)))

def report(oof):
    r = {'non_zero': (rmse(y[nz], oof[nz]), pear(y[nz], oof[nz]))}
    for g in ['dur_45s', 'dur_60s']:
        m = nz & (grp == g)
        r[g] = (rmse(y[m], oof[m]), pear(y[m], oof[m]))
    r['test_weighted'] = (wrmse(y[nz], oof[nz], w[nz]), wpear(y[nz], oof[nz], w[nz]))
    return pd.DataFrame(r, index=['RMSE', 'Pearson']).T.round(4)

def score(oof):
    r = report(oof)
    rm, pe = r.loc['test_weighted', 'RMSE'], r.loc['test_weighted', 'Pearson']
    return dict(rmse_w=rm, pear_w=pe, proxy=(rm + 1 - pe) / 2,
                pear_45=r.loc['dur_45s', 'Pearson'], pear_60=r.loc['dur_60s', 'Pearson'])

print('train rows:', len(folds), '| non-zero:', int(nz.sum()), '| test rows:', len(test))
print('wavlm L21:', W_tr.shape, '| whisper L30:', H_tr.shape, '| text:', TF_tr.shape)

In [ ]:
# S2: Base models
class Winsor(BaseEstimator, TransformerMixin):
    def __init__(self, lo=1, hi=99):
        self.lo, self.hi = lo, hi
    def fit(self, X, y=None):
        self.lo_ = np.percentile(X, self.lo, axis=0)
        self.hi_ = np.percentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lo_, self.hi_)

def ridge_wavlm():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 5, 9)))

def ridge_whisper():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 6, 11)))

def ridge_text():
    return make_pipeline(SimpleImputer(strategy='median'), Winsor(), StandardScaler(),
                         RidgeCV(alphas=np.logspace(-1, 4, 12)))

FOLD_COLS = ['g250', 'g120']
R, rows = {}, []

def run(name, X_tr, X_te, make_model):
    X_tr, X_te = np.asarray(X_tr, dtype=np.float32), np.asarray(X_te, dtype=np.float32)
    oof = {}
    for col in FOLD_COLS:
        o = np.full(len(y), np.nan)
        for f in range(5):
            va = (folds[col] == f).values
            m = make_model().fit(X_tr[(~va) & nz], y[(~va) & nz])
            o[va & nz] = np.clip(m.predict(X_tr[va & nz]), 1, 5)
        oof[col] = o
    m = make_model().fit(X_tr[nz], y[nz])
    R[name] = dict(oof=oof, test=np.clip(m.predict(X_te), 1, 5))
    s, s2 = score(oof['g250']), score(oof['g120'])
    return dict(model=name, proxy_g250=s['proxy'], proxy_g120=s2['proxy'], rmse_w=s['rmse_w'],
                pear_w=s['pear_w'], pear_45=s['pear_45'], pear_60=s['pear_60'],
                train_rmse=rmse(y[nz], np.clip(m.predict(X_tr[nz]), 1, 5)))

cols = TF.columns.tolist()
conf = [c for c in cols if c.startswith(('wp_', 'seg_logprob', 'max_no_speech'))]
noconf = [c for c in cols if c not in conf]

rows.append(run('wavlm_L21_ridge', W_tr, W_te, ridge_wavlm))
rows.append(run('whisper_L30_ridge', H_tr, H_te, ridge_whisper))
rows.append(run('text_all_ridge', TF_tr[cols], TF_te[cols], ridge_text))
rows.append(run('text_noconf_ridge', TF_tr[noconf], TF_te[noconf], ridge_text))
tab = pd.DataFrame(rows).round(4)
print(tab.sort_values('proxy_g250').to_string(index=False))

so = pd.read_csv(find('oof_wavlm_g250.csv')).set_index('uid').oof_g250.reindex(folds.uid).values
print('\nWavLM oof vs notebook 5: max abs diff', round(float(np.nanmax(np.abs(so - R['wavlm_L21_ridge']['oof']['g250']))), 5))

In [ ]:
def avg_oof(names, col): return np.mean([R[n]['oof'][col] for n in names], axis=0)
def avg_test(names): return np.mean([R[n]['test'] for n in names], axis=0)

def blend_cv(names, col, alpha=1.0):
    P = np.column_stack([R[n]['oof'][col] for n in names])
    out = np.full(len(y), np.nan)
    for f in range(5):
        va = (folds[col] == f).values
        m = Ridge(alpha=alpha, positive=True).fit(P[(~va) & nz], y[(~va) & nz])
        out[va & nz] = np.clip(m.predict(P[va & nz]), 1, 5)
    return out

def blend_fit(names, alpha=1.0, col='g250'):
    P = np.column_stack([R[n]['oof'][col] for n in names])
    T = np.column_stack([R[n]['test'] for n in names])
    m = Ridge(alpha=alpha, positive=True).fit(P[nz], y[nz])
    print('  weights', dict(zip(names, np.round(m.coef_, 3))), '| intercept', round(float(m.intercept_), 3))
    return np.clip(m.predict(T), 1, 5)

W, H, T1, T2 = 'wavlm_L21_ridge', 'whisper_L30_ridge', 'text_all_ridge', 'text_noconf_ridge'
sets = {'W+H': [W, H], 'W+H+text_all': [W, H, T1], 'W+H+text_noconf': [W, H, T2], 'all4': [W, H, T1, T2]}

cands = []
for sname, names in sets.items():
    for kind in ['avg', 'fit']:
        o = {col: (avg_oof(names, col) if kind == 'avg' else blend_cv(names, col)) for col in FOLD_COLS}
        s, s2 = score(o['g250']), score(o['g120'])
        cands.append(dict(name=f'{kind}|{sname}', kind=kind, n=len(names), names=names, oof=o,
                          proxy_g250=s['proxy'], proxy_g120=s2['proxy'], rmse_w=s['rmse_w'],
                          pear_w=s['pear_w'], pear_45=s['pear_45'], pear_60=s['pear_60']))
res = pd.DataFrame([{k: v for k, v in c.items() if k not in ('names', 'oof')} for c in cands])
res['proxy_mean'] = (res.proxy_g250 + res.proxy_g120) / 2
print(res.drop(columns=['kind', 'n']).round(4).sort_values('proxy_mean').to_string(index=False))

# simplicity rule: among candidates within 0.005 of the best mean proxy, take the simplest
best = res.proxy_mean.min()
ok = res[res.proxy_mean <= best + 0.005].copy()
ok['is_fit'] = (ok.kind == 'fit').astype(int)
pick = ok.sort_values(['is_fit', 'n', 'proxy_mean']).iloc[0]
ch = next(c for c in cands if c['name'] == pick['name'])
print('\nbest mean proxy:', round(best, 4), '| chosen (simplest within 0.005):', ch['name'])
print(report(ch['oof']['g250']))
print('reference: earlier WavLM+text blend (notebook 7) g250 proxy 0.4000, whisper alone 0.3971')

pred_te = avg_test(ch['names']) if ch['kind'] == 'avg' else blend_fit(ch['names'])
print('test pred summary:', pd.Series(pred_te).describe().round(3).to_dict())
print('corr with whisper-only test preds:', round(pear(pred_te, R[H]['test']), 4))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
allp = pd.concat([tab[['model', 'proxy_g250']].rename(columns={'model': 'name'}),
                  res[['name', 'proxy_g250']]]).sort_values('proxy_g250')
ax[0].barh(allp.name, allp.proxy_g250)
ax[0].set_xlim(0.36, allp.proxy_g250.max() + 0.01)
ax[0].set_title('CV proxy on g250 folds (lower is better, axis starts at 0.36)')
sns.scatterplot(x=y[nz], y=ch['oof']['g250'][nz], hue=grp[nz], alpha=0.5, ax=ax[1])
ax[1].plot([1, 5], [1, 5], 'r--'); ax[1].set_xlabel('true'); ax[1].set_ylabel('OOF prediction')
ax[1].set_title('Chosen model: predicted vs true')
plt.tight_layout(); plt.show()

bp = pd.DataFrame({'uid': folds.uid})
for n in R:
    for col in FOLD_COLS:
        bp[f'{n}__{col}'] = R[n]['oof'][col]
bp.to_csv(f'{WORK}/base_oof.csv', index=False)
pd.DataFrame({'filename': test.filename, **{n: R[n]['test'] for n in R}}).to_csv(f'{WORK}/base_test.csv', index=False)
pd.DataFrame({'filename': test.filename, 'label': pred_te}).to_csv(f'{WORK}/submission_stack_v2.csv', index=False)
print('saved submission_stack_v2.csv, base_oof.csv, base_test.csv')